# Candidate ReRank Model using Handcrafted Rules
In this notebook, we present a "candidate rerank" model using handcrafted rules. We can improve this model by engineering features, merging them unto items and users, and training a reranker model (such as XGB) to choose our final 20. Furthermore to tune and improve this notebook, we should build a local CV scheme to experiment new logic and/or models.

UPDATE: I published a notebook to compute validation score [here][10] using Radek's scheme described [here][11].

Note in this competition, a "session" actually means a unique "user". So our task is to predict what each of the `1,671,803` test "users" (i.e. "sessions") will do in the future. For each test "user" (i.e. "session") we must predict what they will `click`, `cart`, and `order` during the remainder of the week long test period.

### Step 1 - Generate Candidates
For each test user, we generate possible choices, i.e. candidates. In this notebook, we generate candidates from 5 sources:
* User history of clicks, carts, orders
* Most popular 20 clicks, carts, orders during test week
* Co-visitation matrix of click/cart/order to cart/order with type weighting
* Co-visitation matrix of cart/order to cart/order called buy2buy
* Co-visitation matrix of click/cart/order to clicks with time weighting

### Step 2 - ReRank and Choose 20
Given the list of candidates, we must select 20 to be our predictions. In this notebook, we do this with a set of handcrafted rules. We can improve our predictions by training an XGBoost model to select for us. Our handcrafted rules give priority to:
* Most recent previously visited items
* Items previously visited multiple times
* Items previously in cart or order
* Co-visitation matrix of cart/order to cart/order
* Current popular items

![](https://raw.githubusercontent.com/cdeotte/Kaggle_Images/main/Nov-2022/c_r_model.png)
  
# Credits
We thank many Kagglers who have shared ideas. We use co-visitation matrix idea from Vladimir [here][1]. We use groupby sort logic from Sinan in comment section [here][4]. We use duplicate prediction removal logic from Radek [here][5]. We use multiple visit logic from Pietro [here][2]. We use type weighting logic from Ingvaras [here][3]. We use leaky test data from my previous notebook [here][4]. And some ideas may have originated from Tawara [here][6] and KJ [here][7]. We use Colum2131's parquets [here][8]. Above image is from Ravi's discussion about candidate rerank models [here][9]

[1]: https://www.kaggle.com/code/vslaykovsky/co-visitation-matrix
[2]: https://www.kaggle.com/code/pietromaldini1/multiple-clicks-vs-latest-items
[3]: https://www.kaggle.com/code/ingvarasgalinskas/item-type-vs-multiple-clicks-vs-latest-items
[4]: https://www.kaggle.com/code/cdeotte/test-data-leak-lb-boost
[5]: https://www.kaggle.com/code/radek1/co-visitation-matrix-simplified-imprvd-logic
[6]: https://www.kaggle.com/code/ttahara/otto-mors-aid-frequency-baseline
[7]: https://www.kaggle.com/code/whitelily/co-occurrence-baseline
[8]: https://www.kaggle.com/datasets/columbia2131/otto-chunk-data-inparquet-format
[9]: https://www.kaggle.com/competitions/otto-recommender-system/discussion/364721
[10]: https://www.kaggle.com/cdeotte/compute-validation-score-cv-564
[11]: https://www.kaggle.com/competitions/otto-recommender-system/discussion/364991

# Notes
Below are notes about versions:
* **Version 1 LB 0.573** Uses popular ideas from public notebooks and adds additional co-visitation matrices and additional logic. Has CV `0.563`. See validation notebook version 2 [here][1].
* **Version 2 LB 573** Refactor logic for `suggest_buys(df)` to make it clear how new co-visitation matrices are reranking the candidates by adding to candidate weights. Also new logic boosts CV by `+0.0003`. Also LB is slightly better too. See validation notebook version 3 [here][1]
* **Version 3** is the same as version 2 but 1.5x faster co-visitation matrix computation!
* **Version 4 LB 575** Use top20 for clicks and top15 for carts and buys (instead of top40 and top40). This boosts CV `+0.0015` hooray! New CV is `0.5647`. See validation version 5 [here][1]
* **Version 5** is the same as version 4 but 2x faster co-visitation matrix computation! (and 3x faster than version 1)
* **Version 6** Stay tuned for more versions...

[1]: https://www.kaggle.com/code/cdeotte/compute-validation-score-cv-564

# Step 1 - Candidate Generation with RAPIDS
For candidate generation, we build three co-visitation matrices. One computes the popularity of cart/order given a user's previous click/cart/order. We apply type weighting to this matrix. One computes the popularity of cart/order given a user's previous cart/order. We call this "buy2buy" matrix. One computes the popularity of clicks given a user previously click/cart/order.  We apply time weighting to this matrix. We will use RAPIDS cuDF GPU to compute these matrices quickly!

In [5]:
VER = 5

import pandas as pd, numpy as np
from tqdm.notebook import tqdm
import os, sys, pickle, glob, gc
from collections import Counter
import cudf, itertools
print('We will use RAPIDS version',cudf.__version__)

We will use RAPIDS version 21.10.01


## Compute Three Co-visitation Matrices with RAPIDS
We will compute 3 co-visitation matrices using RAPIDS cuDF on GPU. This is 30x faster than using Pandas CPU like other public notebooks! For maximum speed, set the variable `DISK_PIECES` to the smallest number possible based on the GPU you are using without incurring memory errors. If you run this code offline with 32GB GPU ram, then you can use `DISK_PIECES = 1` and compute each co-visitation matrix in almost 1 minute! Kaggle's GPU only has 16GB ram, so we use `DISK_PIECES = 4` and it takes an amazing 3 minutes each! Below are some of the tricks to speed up computation
* Use RAPIDS cuDF GPU instead of Pandas CPU
* Read disk once and save in CPU RAM for later GPU multiple use
* Process largest amount of data possible on GPU at one time
* Merge data in two stages. Multiple small to single medium. Multiple medium to single large.
* Write result as parquet instead of dictionary

In [6]:
%%time
# CACHE FUNCTIONS
def read_file(f):
    return cudf.DataFrame( data_cache[f] )
def read_file_to_cache(f):
    df = pd.read_parquet(f)
    df.ts = (df.ts/1000).astype('int32')
    df['type'] = df['type'].map(type_labels).astype('int8')
    return df

# CACHE THE DATA ON CPU BEFORE PROCESSING ON GPU
data_cache = {}
type_labels = {'clicks':0, 'carts':1, 'orders':2}
files = glob.glob('../input/otto-chunk-data-inparquet-format/*_parquet/*')
for f in files: data_cache[f] = read_file_to_cache(f)

# CHUNK PARAMETERS
READ_CT = 5
CHUNK = int( np.ceil( len(files)/6 ))
print(f'We will process {len(files)} files, in groups of {READ_CT} and chunks of {CHUNK}.')

We will process 146 files, in groups of 5 and chunks of 25.
CPU times: user 48.4 s, sys: 10.3 s, total: 58.7 s
Wall time: 58.6 s


## 1) "Carts Orders" Co-visitation Matrix - Type Weighted

In [7]:
%%time
type_weight = {0:1, 1:6, 2:3}

# USE SMALLEST DISK_PIECES POSSIBLE WITHOUT MEMORY ERROR
DISK_PIECES = 4
SIZE = 1.86e6/DISK_PIECES

# COMPUTE IN PARTS FOR MEMORY MANGEMENT
for PART in range(DISK_PIECES):
    print()
    print('### DISK PART',PART+1)
    
    # MERGE IS FASTEST PROCESSING CHUNKS WITHIN CHUNKS
    # => OUTER CHUNKS
    for j in range(6):
        a = j*CHUNK
        b = min( (j+1)*CHUNK, len(files) )
        print(f'Processing files {a} thru {b-1} in groups of {READ_CT}...')
        
        # => INNER CHUNKS
        for k in range(a,b,READ_CT):
            # READ FILE
            df = [read_file(files[k])]
            for i in range(1,READ_CT): 
                if k+i<b: df.append( read_file(files[k+i]) )
            df = cudf.concat(df,ignore_index=True,axis=0)
            df = df.sort_values(['session','ts'],ascending=[True,False])
            # USE TAIL OF SESSION
            df = df.reset_index(drop=True)
            df['n'] = df.groupby('session').cumcount()
            df = df.loc[df.n<30].drop('n',axis=1)
            # CREATE PAIRS
            df = df.merge(df,on='session')
            df = df.loc[ ((df.ts_x - df.ts_y).abs()< 24 * 60 * 60) & (df.aid_x != df.aid_y) ]
            # MEMORY MANAGEMENT COMPUTE IN PARTS
            df = df.loc[(df.aid_x >= PART*SIZE)&(df.aid_x < (PART+1)*SIZE)]
            # ASSIGN WEIGHTS
            df = df[['session', 'aid_x', 'aid_y','type_y']].drop_duplicates(['session', 'aid_x', 'aid_y'])
            df['wgt'] = df.type_y.map(type_weight)
            df = df[['aid_x','aid_y','wgt']]
            df.wgt = df.wgt.astype('float32')
            df = df.groupby(['aid_x','aid_y']).wgt.sum()
            # COMBINE INNER CHUNKS
            if k==a: tmp2 = df
            else: tmp2 = tmp2.add(df, fill_value=0)
            print(k,', ',end='')
        print()
        # COMBINE OUTER CHUNKS
        if a==0: tmp = tmp2
        else: tmp = tmp.add(tmp2, fill_value=0)
        del tmp2, df
        gc.collect()
    # CONVERT MATRIX TO DICTIONARY
    tmp = tmp.reset_index()
    tmp = tmp.sort_values(['aid_x','wgt'],ascending=[True,False])
    # SAVE TOP 40
    tmp = tmp.reset_index(drop=True)
    tmp['n'] = tmp.groupby('aid_x').aid_y.cumcount()
    tmp = tmp.loc[tmp.n<15].drop('n',axis=1)
    # SAVE PART TO DISK (convert to pandas first uses less memory)
    tmp.to_pandas().to_parquet(f'top_15_carts_orders_v{VER}_{PART}.pqt')


### DISK PART 1
Processing files 0 thru 24 in groups of 5...


/opt/conda/lib/python3.7/site-packages/cudf/core/frame.py:2600: UserWarning: When using a sequence of booleans for `ascending`, `na_position` flag is not yet supported and defaults to treating nulls as greater than all numbers
  "When using a sequence of booleans for `ascending`, "


0 , 5 , 10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 35 , 40 , 45 , 
Processing files 50 thru 74 in groups of 5...
50 , 55 , 60 , 65 , 70 , 
Processing files 75 thru 99 in groups of 5...
75 , 80 , 85 , 90 , 95 , 
Processing files 100 thru 124 in groups of 5...
100 , 105 , 110 , 115 , 120 , 
Processing files 125 thru 145 in groups of 5...
125 , 130 , 135 , 140 , 145 , 

### DISK PART 2
Processing files 0 thru 24 in groups of 5...
0 , 5 , 10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 35 , 40 , 45 , 
Processing files 50 thru 74 in groups of 5...
50 , 55 , 60 , 65 , 70 , 
Processing files 75 thru 99 in groups of 5...
75 , 80 , 85 , 90 , 95 , 
Processing files 100 thru 124 in groups of 5...
100 , 105 , 110 , 115 , 120 , 
Processing files 125 thru 145 in groups of 5...
125 , 130 , 135 , 140 , 145 , 

### DISK PART 3
Processing files 0 thru 24 in groups of 5...
0 , 5 , 10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 

## 2) "Buy2Buy" Co-visitation Matrix

In [8]:
%%time
# USE SMALLEST DISK_PIECES POSSIBLE WITHOUT MEMORY ERROR
DISK_PIECES = 1
SIZE = 1.86e6/DISK_PIECES

# COMPUTE IN PARTS FOR MEMORY MANGEMENT
for PART in range(DISK_PIECES):
    print()
    print('### DISK PART',PART+1)
    
    # MERGE IS FASTEST PROCESSING CHUNKS WITHIN CHUNKS
    # => OUTER CHUNKS
    for j in range(6):
        a = j*CHUNK
        b = min( (j+1)*CHUNK, len(files) )
        print(f'Processing files {a} thru {b-1} in groups of {READ_CT}...')
        
        # => INNER CHUNKS
        for k in range(a,b,READ_CT):
            # READ FILE
            df = [read_file(files[k])]
            for i in range(1,READ_CT): 
                if k+i<b: df.append( read_file(files[k+i]) )
            df = cudf.concat(df,ignore_index=True,axis=0)
            df = df.loc[df['type'].isin([1,2])] # ONLY WANT CARTS AND ORDERS
            df = df.sort_values(['session','ts'],ascending=[True,False])
            # USE TAIL OF SESSION
            df = df.reset_index(drop=True)
            df['n'] = df.groupby('session').cumcount()
            df = df.loc[df.n<30].drop('n',axis=1)
            # CREATE PAIRS
            df = df.merge(df,on='session')
            df = df.loc[ ((df.ts_x - df.ts_y).abs()< 14 * 24 * 60 * 60) & (df.aid_x != df.aid_y) ] # 14 DAYS
            # MEMORY MANAGEMENT COMPUTE IN PARTS
            df = df.loc[(df.aid_x >= PART*SIZE)&(df.aid_x < (PART+1)*SIZE)]
            # ASSIGN WEIGHTS
            df = df[['session', 'aid_x', 'aid_y','type_y']].drop_duplicates(['session', 'aid_x', 'aid_y'])
            df['wgt'] = 1
            df = df[['aid_x','aid_y','wgt']]
            df.wgt = df.wgt.astype('float32')
            df = df.groupby(['aid_x','aid_y']).wgt.sum()
            # COMBINE INNER CHUNKS
            if k==a: tmp2 = df
            else: tmp2 = tmp2.add(df, fill_value=0)
            print(k,', ',end='')
        print()
        # COMBINE OUTER CHUNKS
        if a==0: tmp = tmp2
        else: tmp = tmp.add(tmp2, fill_value=0)
        del tmp2, df
        gc.collect()
    # CONVERT MATRIX TO DICTIONARY
    tmp = tmp.reset_index()
    tmp = tmp.sort_values(['aid_x','wgt'],ascending=[True,False])
    # SAVE TOP 40
    tmp = tmp.reset_index(drop=True)
    tmp['n'] = tmp.groupby('aid_x').aid_y.cumcount()
    tmp = tmp.loc[tmp.n<15].drop('n',axis=1)
    # SAVE PART TO DISK (convert to pandas first uses less memory)
    tmp.to_pandas().to_parquet(f'top_15_buy2buy_v{VER}_{PART}.pqt')


### DISK PART 1
Processing files 0 thru 24 in groups of 5...
0 , 5 , 

/opt/conda/lib/python3.7/site-packages/cudf/core/frame.py:2600: UserWarning: When using a sequence of booleans for `ascending`, `na_position` flag is not yet supported and defaults to treating nulls as greater than all numbers
  "When using a sequence of booleans for `ascending`, "


10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 35 , 40 , 45 , 
Processing files 50 thru 74 in groups of 5...
50 , 55 , 60 , 65 , 70 , 
Processing files 75 thru 99 in groups of 5...
75 , 80 , 85 , 90 , 95 , 
Processing files 100 thru 124 in groups of 5...
100 , 105 , 110 , 115 , 120 , 
Processing files 125 thru 145 in groups of 5...
125 , 130 , 135 , 140 , 145 , 
CPU times: user 21 s, sys: 8.68 s, total: 29.7 s
Wall time: 30 s


## 3) "Clicks" Co-visitation Matrix - Time Weighted

In [9]:
%%time
# USE SMALLEST DISK_PIECES POSSIBLE WITHOUT MEMORY ERROR
DISK_PIECES = 4
SIZE = 1.86e6/DISK_PIECES

# COMPUTE IN PARTS FOR MEMORY MANGEMENT
for PART in range(DISK_PIECES):
    print()
    print('### DISK PART',PART+1)
    
    # MERGE IS FASTEST PROCESSING CHUNKS WITHIN CHUNKS
    # => OUTER CHUNKS
    for j in range(6):
        a = j*CHUNK
        b = min( (j+1)*CHUNK, len(files) )
        print(f'Processing files {a} thru {b-1} in groups of {READ_CT}...')
        
        # => INNER CHUNKS
        for k in range(a,b,READ_CT):
            # READ FILE
            df = [read_file(files[k])]
            for i in range(1,READ_CT): 
                if k+i<b: df.append( read_file(files[k+i]) )
            df = cudf.concat(df,ignore_index=True,axis=0)
            df = df.sort_values(['session','ts'],ascending=[True,False])
            # USE TAIL OF SESSION
            df = df.reset_index(drop=True)
            df['n'] = df.groupby('session').cumcount()
            df = df.loc[df.n<30].drop('n',axis=1)
            # CREATE PAIRS
            df = df.merge(df,on='session')
            df = df.loc[ ((df.ts_x - df.ts_y).abs()< 24 * 60 * 60) & (df.aid_x != df.aid_y) ]
            # MEMORY MANAGEMENT COMPUTE IN PARTS
            df = df.loc[(df.aid_x >= PART*SIZE)&(df.aid_x < (PART+1)*SIZE)]
            # ASSIGN WEIGHTS
            df = df[['session', 'aid_x', 'aid_y','ts_x']].drop_duplicates(['session', 'aid_x', 'aid_y'])
            df['wgt'] = 1 + 3*(df.ts_x - 1659304800)/(1662328791-1659304800)
            df = df[['aid_x','aid_y','wgt']]
            df.wgt = df.wgt.astype('float32')
            df = df.groupby(['aid_x','aid_y']).wgt.sum()
            # COMBINE INNER CHUNKS
            if k==a: tmp2 = df
            else: tmp2 = tmp2.add(df, fill_value=0)
            print(k,', ',end='')
        print()
        # COMBINE OUTER CHUNKS
        if a==0: tmp = tmp2
        else: tmp = tmp.add(tmp2, fill_value=0)
        del tmp2, df
        gc.collect()
    # CONVERT MATRIX TO DICTIONARY
    tmp = tmp.reset_index()
    tmp = tmp.sort_values(['aid_x','wgt'],ascending=[True,False])
    # SAVE TOP 40
    tmp = tmp.reset_index(drop=True)
    tmp['n'] = tmp.groupby('aid_x').aid_y.cumcount()
    tmp = tmp.loc[tmp.n<20].drop('n',axis=1)
    # SAVE PART TO DISK (convert to pandas first uses less memory)
    tmp.to_pandas().to_parquet(f'top_20_clicks_v{VER}_{PART}.pqt')


### DISK PART 1
Processing files 0 thru 24 in groups of 5...
0 , 5 , 10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 35 , 40 , 45 , 
Processing files 50 thru 74 in groups of 5...
50 , 55 , 60 , 65 , 70 , 
Processing files 75 thru 99 in groups of 5...
75 , 80 , 85 , 90 , 95 , 
Processing files 100 thru 124 in groups of 5...
100 , 105 , 110 , 115 , 120 , 
Processing files 125 thru 145 in groups of 5...
125 , 130 , 135 , 140 , 145 , 

### DISK PART 2
Processing files 0 thru 24 in groups of 5...
0 , 5 , 10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 35 , 40 , 45 , 
Processing files 50 thru 74 in groups of 5...
50 , 55 , 60 , 65 , 70 , 
Processing files 75 thru 99 in groups of 5...
75 , 80 , 85 , 90 , 95 , 
Processing files 100 thru 124 in groups of 5...
100 , 105 , 110 , 115 , 120 , 
Processing files 125 thru 145 in groups of 5...
125 , 130 , 135 , 140 , 145 , 

### DISK PART 3
Processing files 0 thru 24 in groups of 5...
0 , 5 , 10 , 15 , 

In [10]:
# FREE MEMORY
del data_cache, tmp
_ = gc.collect()

# Step 2 - ReRank (choose 20) using handcrafted rules
For description of the handcrafted rules, read this notebook's intro.

In [11]:
def load_test():    
    dfs = []
    for e, chunk_file in enumerate(glob.glob('../input/otto-chunk-data-inparquet-format/test_parquet/*')):
        chunk = pd.read_parquet(chunk_file)
        chunk.ts = (chunk.ts/1000).astype('int32')
        chunk['type'] = chunk['type'].map(type_labels).astype('int8')
        dfs.append(chunk)
    return pd.concat(dfs).reset_index(drop=True) #.astype({"ts": "datetime64[ms]"})

test_df = load_test()
print('Test data has shape',test_df.shape)
test_df.head()

Test data has shape (6928123, 4)


,session,aid,ts,type
0,13099779,245308,1661795832,0
1,13099779,245308,1661795862,1
2,13099779,972319,1661795888,0
3,13099779,972319,1661795898,1
4,13099779,245308,1661795907,0


In [12]:
def load_train():    
    dfs = []
    for e, chunk_file in enumerate(glob.glob('../input/otto-chunk-data-inparquet-format/train_parquet/*')):
        chunk = pd.read_parquet(chunk_file)
        chunk.ts = (chunk.ts/1000).astype('int32')
        chunk['type'] = chunk['type'].map(type_labels).astype('int8')
        dfs.append(chunk)
    return pd.concat(dfs).reset_index(drop=True) #.astype({"ts": "datetime64[ms]"})

train_df = load_train()
print('Test data has shape',train_df.shape)
train_df.head()

Test data has shape (216716096, 4)


,session,aid,ts,type
0,12800000,169484,1661701761,0
1,12800000,979081,1661708105,0
2,12800000,979081,1661708169,0
3,12800000,979081,1661710002,0
4,12800001,1110494,1661701761,0


In [ ]:
# %%time
# def pqt_to_dict(df):
#     return df.groupby('aid_x').aid_y.apply(list).to_dict()
# # LOAD THREE CO-VISITATION MATRICES
# top_20_clicks = pqt_to_dict( pd.read_parquet(f'top_20_clicks_v{VER}_0.pqt') )
# for k in range(1,DISK_PIECES): 
#     top_20_clicks.update( pqt_to_dict( pd.read_parquet(f'top_20_clicks_v{VER}_{k}.pqt') ) )
# top_20_buys = pqt_to_dict( pd.read_parquet(f'top_15_carts_orders_v{VER}_0.pqt') )
# for k in range(1,DISK_PIECES): 
#     top_20_buys.update( pqt_to_dict( pd.read_parquet(f'top_15_carts_orders_v{VER}_{k}.pqt') ) )
# top_20_buy2buy = pqt_to_dict( pd.read_parquet(f'top_15_buy2buy_v{VER}_0.pqt') )

# # TOP CLICKS AND ORDERS IN TEST
# top_clicks = test_df.loc[test_df['type']=='clicks','aid'].value_counts().index.values[:20]
# top_orders = test_df.loc[test_df['type']=='orders','aid'].value_counts().index.values[:20]

# print('Here are size of our 3 co-visitation matrices:')
# print( len( top_20_clicks ), len( top_20_buy2buy ), len( top_20_buys ) )

In [13]:
import numpy as np
import pandas as pd
import gc, glob
from tqdm import tqdm
import xgboost as xgb

In [15]:
def pqt_to_dict(df):
    return df.groupby('aid_x').aid_y.apply(list).to_dict()

top_20_clicks = {}
for k in range(DISK_PIECES):
    top_20_clicks.update(
        pqt_to_dict(pd.read_parquet(f'top_20_clicks_v{VER}_{k}.pqt'))
    )

top_20_buys = {}
for k in range(DISK_PIECES):
    top_20_buys.update(
        pqt_to_dict(pd.read_parquet(f'top_15_carts_orders_v{VER}_{k}.pqt'))
    )

top_20_buy2buy = pqt_to_dict(
    pd.read_parquet(f'top_15_buy2buy_v{VER}_0.pqt')
)

In [32]:
FEATURE_COLS = [
    'cnt', 'recency', 'max_type',
    'in_click_cov', 'in_buy_cov', 'in_buy2buy'
]

def generate_candidates(hist_aids, max_candidates=100):
    recents = list(dict.fromkeys(hist_aids[::-1]))[:20]
    covisit = []

    for a in recents:
        covisit += top_20_clicks.get(a, [])
        covisit += top_20_buys.get(a, [])
        covisit += top_20_buy2buy.get(a, [])

    return list(dict.fromkeys(recents + covisit))[:max_candidates]


def make_features(hist_df, candidates):
    last_ts = hist_df['ts'].max()

    aid_counts = hist_df['aid'].value_counts()
    aid_last_ts = hist_df.groupby('aid')['ts'].max()
    aid_type_max = hist_df.groupby('aid')['type'].max()

    X = np.zeros((len(candidates), 6), dtype=np.float32)

    for i, aid in enumerate(candidates):
        X[i, 0] = aid_counts.get(aid, 0)
        X[i, 1] = last_ts - aid_last_ts.get(aid, last_ts)
        X[i, 2] = aid_type_max.get(aid, -1)
        X[i, 3] = aid in top_20_clicks
        X[i, 4] = aid in top_20_buys
        X[i, 5] = aid in top_20_buy2buy

    return X


def make_labels(candidates, future_aids):
    return np.fromiter(
        (aid in future_aids for aid in candidates),
        dtype=np.int8,
        count=len(candidates)
    )

In [33]:
import numpy as np
import pandas as pd
import glob, os, gc
import xgboost as xgb

os.makedirs("train_chunks", exist_ok=True)

MAX_SESSIONS_PER_CHUNK = 250_000  # adjust so each chunk fits in memory
chunk_id = 0

X_buf, y_buf, group_buf = [], [], []
session_count = 0

train_files = glob.glob('../input/otto-chunk-data-inparquet-format/train_parquet/*')

def flush_chunk(X_buf, y_buf, group_buf, chunk_id):
    X_chunk = np.vstack(X_buf)
    y_chunk = np.concatenate(y_buf)
    group_chunk = np.array(group_buf, dtype=np.int32)
    np.savez_compressed(f"train_chunks/chunk_{chunk_id}.npz",
                        X=X_chunk, y=y_chunk, group=group_chunk)
    # print(f"Saved chunk {chunk_id}, shape {X_chunk.shape}")
    X_buf.clear(); y_buf.clear(); group_buf.clear()
    del X_chunk, y_chunk, group_chunk
    gc.collect()

In [34]:
NUM_FEATURES = 6

In [ ]:
import cudf, pandas as pd, numpy as np, gc, glob, os
from tqdm import tqdm
import xgboost as xgb

VER = 5
DISK_PIECES = 4  # adjust based on GPU memory
MAX_CANDIDATES = 100  # max candidates per session
FEATURE_COLS = ['cnt','recency','max_type','in_click_cov','in_buy_cov','in_buy2buy']
NUM_FEATURES = len(FEATURE_COLS)
MAX_SESSIONS_PER_CHUNK = 250_000  # tune to fit GPU memory

os.makedirs("train_chunks", exist_ok=True)

# # load co-visitation matrices (already computed with GPU)
# def pqt_to_dict(df):
#     return df.groupby('aid_x').aid_y.apply(list).to_dict()

# top_20_clicks = {}
# top_20_buys = {}
# top_20_buy2buy = {}
# for k in range(DISK_PIECES):
#     top_20_clicks.update(pqt_to_dict(pd.read_parquet(f'top_20_clicks_v{VER}_{k}.pqt')))
#     top_20_buys.update(pqt_to_dict(pd.read_parquet(f'top_15_carts_orders_v{VER}_{k}.pqt')))
#     top_20_buy2buy.update(pqt_to_dict(pd.read_parquet(f'top_15_buy2buy_v{VER}_{k}.pqt')))

# GPU-friendly generate_candidates
def generate_candidates(hist_aids):
    recents = list(dict.fromkeys(hist_aids[::-1]))[:20]
    covisit = []
    for a in recents:
        covisit += top_20_clicks.get(a, [])
        covisit += top_20_buys.get(a, [])
        covisit += top_20_buy2buy.get(a, [])
    candidates = list(dict.fromkeys(recents + covisit))
    return candidates[:MAX_CANDIDATES]

# GPU-friendly feature generation
def make_features(hist_df, candidates):
    last_ts = hist_df['ts'].max()
    aid_counts = hist_df['aid'].value_counts()
    aid_last_ts = hist_df.groupby('aid')['ts'].max()
    aid_type_max = hist_df.groupby('aid')['type'].max()
    feats = []
    for aid in candidates:
        feats.append([
            aid_counts.get(aid,0),
            last_ts - aid_last_ts.get(aid,last_ts),
            aid_type_max.get(aid,-1),
            int(aid in top_20_clicks),
            int(aid in top_20_buys),
            int(aid in top_20_buy2buy)
        ])
    return np.array(feats, dtype=np.float32)

def make_labels(candidates, future_aids):
    return np.array([1 if aid in future_aids else 0 for aid in candidates], dtype=np.int8)

def flush_chunk(X_buf, y_buf, group_buf, chunk_id):
    np.savez_compressed(f"train_chunks/chunk_{chunk_id}.npz",
                        X=X_buf, y=y_buf, group=group_buf)
    X_buf.fill(0); y_buf.fill(0); group_buf.fill(0)
    gc.collect()

# Read train files in chunks on GPU
train_files = glob.glob('../input/otto-chunk-data-inparquet-format/train_parquet/*')

for PART in range(DISK_PIECES):
    print(f"\nProcessing PART {PART+1}/{DISK_PIECES}")
    
    # preallocate buffers
    X_buf = np.zeros((MAX_SESSIONS_PER_CHUNK * MAX_CANDIDATES, NUM_FEATURES), dtype=np.float32)
    y_buf = np.zeros((MAX_SESSIONS_PER_CHUNK * MAX_CANDIDATES,), dtype=np.int8)
    group_buf = np.zeros((MAX_SESSIONS_PER_CHUNK,), dtype=np.int32)
    buf_ptr = 0
    session_ptr = 0
    chunk_id = 0
    
    for f in tqdm(train_files, desc="Files"):
        # load chunk to GPU
        df = cudf.read_parquet(f)
        df.ts = (df.ts // 1000).astype('int32')
        df['type'] = df['type'].map({'clicks':0,'carts':1,'orders':2}).astype('int8')

        # process only aid range for this PART
        aid_min = PART * 1.86e6 / DISK_PIECES
        aid_max = (PART+1) * 1.86e6 / DISK_PIECES
        df = df[(df['aid'] >= aid_min) & (df['aid'] < aid_max)]

        # sort per session
        df = df.sort_values(['session','ts'], ascending=[True,False])
        df['n'] = df.groupby('session').cumcount()
        df = df[df.n < 30].drop('n', axis=1)

        sessions_in_file = df['session'].nunique()
        for session, df_sess in df.groupby('session', sort=False):
            if len(df_sess) < 3:
                continue

            split = int(len(df_sess) * 0.7)
            hist = df_sess.iloc[:split].to_pandas()
            future = df_sess.iloc[split:].to_pandas()
            future_aids = set(future['aid'].values)

            candidates = generate_candidates(hist['aid'].values)
            X_sess = make_features(hist, candidates)
            y_sess = make_labels(candidates, future_aids)
            n_cand = len(candidates)

            X_buf[buf_ptr:buf_ptr+n_cand, :] = X_sess
            y_buf[buf_ptr:buf_ptr+n_cand] = y_sess
            group_buf[session_ptr] = n_cand

            buf_ptr += n_cand
            session_ptr += 1

            if session_ptr == MAX_SESSIONS_PER_CHUNK:
                flush_chunk(X_buf[:buf_ptr,:], y_buf[:buf_ptr], group_buf[:session_ptr], chunk_id)
                buf_ptr = 0
                session_ptr = 0
                chunk_id += 1

        del df
        gc.collect()

    if session_ptr > 0:
        flush_chunk(X_buf[:buf_ptr,:], y_buf[:buf_ptr], group_buf[:session_ptr], chunk_id)


Processing PART 1/4


Files:   0%|          | 0/129 [00:00<?, ?it/s]/opt/conda/lib/python3.7/site-packages/cudf/core/frame.py:2600: UserWarning: When using a sequence of booleans for `ascending`, `na_position` flag is not yet supported and defaults to treating nulls as greater than all numbers
  "When using a sequence of booleans for `ascending`, "
Files:  19%|█▉        | 25/129 [33:21<2:10:19, 75.19s/it]

In [ ]:
model = xgb.XGBRanker(
    objective='rank:pairwise',
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='gpu_hist',
    predictor='gpu_predictor',
    random_state=42
)

chunk_files = sorted(glob.glob("train_chunks/chunk_*.npz"))

for f in tqdm(chunk_files):
    data = np.load(f)
    X_chunk = data['X']
    y_chunk = data['y']
    group_chunk = data['group']
    
    model.fit(
        X_chunk, y_chunk, group=group_chunk,
        xgb_model=model if chunk_files.index(f) > 0 else None,
        verbose=True
    )
    
    del X_chunk, y_chunk, group_chunk, data
    gc.collect()

In [ ]:
test_files = glob.glob('../input/otto-chunk-data-inparquet-format/test_parquet/*')
preds = {}

for f in tqdm(test_files):
    df = pd.read_parquet(f)
    df.ts = (df.ts / 1000).astype('int32')
    df['type'] = df['type'].map({'clicks':0,'carts':1,'orders':2}).astype('int8')

    for session, df_sess in df.groupby('session', sort=False):
        candidates = generate_candidates(df_sess['aid'].values)
        X = make_features(df_sess, candidates)
        scores = model.predict(X)
        top_idx = np.argsort(-scores)[:20]
        preds[session] = np.array(candidates)[top_idx]

    del df
    gc.collect()

In [ ]:
rows = []

for session, aids in preds.items():
    labels = " ".join(map(str, aids))
    rows.append((f"{session}_clicks", labels))
    rows.append((f"{session}_carts", labels))
    rows.append((f"{session}_orders", labels))

submission = pd.DataFrame(rows, columns=["session_type", "labels"])
submission.to_csv("submission.csv", index=False)

In [ ]:
X_train = np.vstack(X_buf)
y_train = np.concatenate(y_buf)
group = np.array(group_buf, dtype=np.int32)

del X_buf, y_buf, group_buf
gc.collect()

model = xgb.XGBRanker(
    objective='rank:pairwise',
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='gpu_hist',
    predictor='gpu_predictor',
    random_state=42
)

model.fit(X_train, y_train, group=group)

In [ ]:
test_files = glob.glob('../input/otto-chunk-data-inparquet-format/test_parquet/*')
preds = {}

for f in tqdm(test_files):
    df = pd.read_parquet(f)
    df.ts = (df.ts / 1000).astype('int32')
    df['type'] = df['type'].map({'clicks':0,'carts':1,'orders':2}).astype('int8')

    for session, df_sess in df.groupby('session', sort=False):
        candidates = generate_candidates(df_sess['aid'].values)
        X = make_features(df_sess, candidates)

        scores = model.predict(X)
        top_idx = np.argsort(-scores)[:20]

        preds[session] = np.array(candidates)[top_idx]

    del df
    gc.collect()

In [ ]:
rows = []

for session, aids in preds.items():
    labels = " ".join(map(str, aids))
    rows.append((f"{session}_clicks", labels))
    rows.append((f"{session}_carts", labels))
    rows.append((f"{session}_orders", labels))

submission = pd.DataFrame(rows, columns=["session_type", "labels"])
submission.to_csv("submission.csv", index=False)

In [20]:
# import pandas as pd
# import numpy as np
# import glob
# import itertools
# import xgboost as xgb
# from collections import Counter
# import os

# type_labels = {'clicks': 0, 'carts': 1, 'orders': 2}
# DISK_PIECES = 4
# VER = 1

In [11]:
# def generate_candidates(hist_aids):
#     unique_aids = list(dict.fromkeys(hist_aids[::-1]))
#     covisit_clicks = list(itertools.chain(*[top_20_clicks.get(a, []) for a in unique_aids if a in top_20_clicks]))
#     covisit_buy = list(itertools.chain(*[top_20_buy2buy.get(a, []) for a in unique_aids if a in top_20_buy2buy]))
#     candidates = list(dict.fromkeys(unique_aids + covisit_clicks + covisit_buy))
#     return candidates

In [53]:
# np.random.seed(42)

# sess_with_buy = (
#     train_df
#     .groupby('session')['type']
#     .max()
#     .loc[lambda x: x >= 1]
#     .index
# )

# keep_sessions = np.random.choice(
#     sess_with_buy,
#     # size=min(300_000, len(sess_with_buy)),
#     size=len(sess_with_buy),
#     replace=False
# )

# train_df = train_df[train_df['session'].isin(keep_sessions)]
# train_df = train_df.sort_values(['session','ts'])

In [21]:
len(train_df)

216716096

In [22]:
def generate_candidates(hist_aids, max_candidates=100):
    recents = list(dict.fromkeys(hist_aids[::-1]))[:20]

    covisit = []
    for a in recents:
        covisit += top_20_clicks.get(a, [])
        covisit += top_20_buys.get(a, [])
        covisit += top_20_buy2buy.get(a, [])

    candidates = list(dict.fromkeys(recents + covisit))
    return candidates[:max_candidates]

def make_features(hist_df, candidates):
    feats = []
    last_ts = hist_df['ts'].max()

    aid_counts = hist_df['aid'].value_counts()
    aid_last_ts = hist_df.groupby('aid')['ts'].max()
    aid_type_max = hist_df.groupby('aid')['type'].max()

    for aid in candidates:
        feats.append([
            aid,
            aid_counts.get(aid, 0),
            last_ts - aid_last_ts.get(aid, last_ts),
            aid_type_max.get(aid, -1),
            aid in top_20_clicks,
            aid in top_20_buys,
            aid in top_20_buy2buy,
        ])

    return pd.DataFrame(
        feats,
        columns=[
            'aid',
            'cnt',
            'recency',
            'max_type',
            'in_click_cov',
            'in_buy_cov',
            'in_buy2buy'
        ]
    )
    
def make_labels(candidates, future_aids):
    return np.array([1 if aid in future_aids else 0 for aid in candidates], dtype=np.int8)

In [23]:
# infer feature dimension
dummy_feats = make_features(
    train_df.iloc[:10],
    generate_candidates(train_df.iloc[:10]['aid'].values)
)
FEATURE_COLS = [c for c in dummy_feats.columns if c != 'aid']
D = len(FEATURE_COLS)

In [24]:
D

6

In [25]:
from tqdm import tqdm
import numpy as np

X_buf, y_buf, group_buf = [], [], []

MAX_SESSIONS_PER_CHUNK = 5000

def flush_chunk(X_buf, y_buf, group_buf, chunk_id):
    X = np.vstack(X_buf).astype(np.float32)
    y = np.concatenate(y_buf).astype(np.int8)
    group = np.array(group_buf, dtype=np.int32)

    np.savez_compressed(
        f"xgb_chunk_{chunk_id}.npz",
        X=X,
        y=y,
        group=group
    )

    X_buf.clear()
    y_buf.clear()
    group_buf.clear()

In [ ]:
chunk_id = 0
session_count = 0

for session, df_sess in tqdm(
        train_df.groupby('session', sort=False),
        total=train_df['session'].nunique()
    ):

    if len(df_sess) < 3:
        continue

    split = int(len(df_sess) * 0.7)
    hist = df_sess.iloc[:split]
    future = df_sess.iloc[split:]
    future_aids = set(future['aid'].values)

    candidates = generate_candidates(hist['aid'].values)
    feats = make_features(hist, candidates)

    X_buf.append(
        feats[FEATURE_COLS].to_numpy(dtype=np.float32)
    )
    y_buf.append(
        make_labels(candidates, future_aids)
    )
    group_buf.append(len(candidates))

    session_count += 1

    if session_count % MAX_SESSIONS_PER_CHUNK == 0:
        flush_chunk(X_buf, y_buf, group_buf, chunk_id)
        chunk_id += 1

# flush remainder
if X_buf:
    flush_chunk(X_buf, y_buf, group_buf, chunk_id)

  0%|          | 0/12899779 [00:00<?, ?it/s]

In [44]:
# from tqdm import tqdm

# X_all, y_all, group = [], [], []

# for session, df_sess in tqdm(
#         train_df.groupby('session'),
#         total=train_df['session'].nunique()
#     ):
#     if len(df_sess) < 3:
#         continue

#     split = int(len(df_sess) * 0.7)
#     hist = df_sess.iloc[:split]
#     future = df_sess.iloc[split:]
#     future_aids = set(future['aid'].values)

#     candidates = generate_candidates(hist['aid'].values)
#     feats = make_features(hist, candidates)
    
#     # Drop 'aid' before appending to training data
#     X_all.append(feats.drop(columns='aid', errors='ignore'))
#     y_all.append(make_labels(candidates, future_aids))
#     group.append(len(candidates))

100%|██████████| 300000/300000 [20:48<00:00, 240.30it/s] 


In [45]:
X_train = pd.concat(X_all, ignore_index=True)
y_train = np.concatenate(y_all)
group = np.array(group)

In [46]:
import xgboost as xgb

model = xgb.XGBRanker(
    objective='rank:pairwise',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42
)

model.fit(
    X_train,
    y_train,
    group=group
)

XGBRanker(base_score=0.5, booster='gbtree', callbacks=None, colsample_bylevel=1,
          colsample_bynode=1, colsample_bytree=0.8, early_stopping_rounds=None,
          enable_categorical=False, eval_metric=None, gamma=0, gpu_id=-1,
          grow_policy='depthwise', importance_type=None,
          interaction_constraints='', learning_rate=0.05, max_bin=256,
          max_cat_to_onehot=4, max_delta_step=0, max_depth=6, max_leaves=0,
          min_child_weight=1, missing=nan, monotone_constraints='()',
          n_estimators=200, n_jobs=0, num_parallel_tree=1, predictor='auto',
          random_state=42, reg_alpha=0, reg_lambda=1, ...)

In [47]:
from tqdm import tqdm

sessions = test_df['session'].values
unique_sessions, session_ptrs = np.unique(sessions, return_index=True)

preds = {}

for i, sess in enumerate(tqdm(unique_sessions, total=len(unique_sessions))):
    start = session_ptrs[i]
    end = session_ptrs[i+1] if i+1 < len(session_ptrs) else len(test_df)

    df_sess = test_df.iloc[start:end]

    candidates = generate_candidates(df_sess['aid'].values)
    X = make_features(df_sess, candidates)
    aids = X['aid'].values
    
    # convert all features to numeric
    X_model = X.drop(columns='aid').copy()
    X_model = X_model.fillna(0)        # optional: handle NaNs
    X_model = X_model.astype(float)     # convert int/bool to float
    
    # prediction
    scores = model.predict(X_model)
    top_idx = np.argsort(-scores)[:20]
    
    preds[sess] = aids[top_idx]

100%|██████████| 1671803/1671803 [3:05:10<00:00, 150.47it/s]  


# Create Submission CSV
Inferring test data with Pandas groupby is slow. We need to accelerate the following code.

In [50]:
import pandas as pd

rows = []

for session, aids in preds.items():
    labels = " ".join(map(str, aids))

    rows.append((f"{session}_clicks", labels))
    rows.append((f"{session}_carts",  labels))
    rows.append((f"{session}_orders", labels))

submission = pd.DataFrame(rows, columns=["session_type", "labels"])
submission.to_csv("submission.csv", index=False)

submission.head()

,session_type,labels
0,12899779_clicks,59625 469285 397451 523135 225209 164098 45290...
1,12899779_carts,59625 469285 397451 523135 225209 164098 45290...
2,12899779_orders,59625 469285 397451 523135 225209 164098 45290...
3,12899780_clicks,1142000 736515 973453 582732 209046 1735169 43...
4,12899780_carts,1142000 736515 973453 582732 209046 1735169 43...
